In [1]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pingouin as pg

sns.set_context('talk')

/home/nicolas/.local/lib/python3.8/site-packages/outdated/utils.py:14: OutdatedPackageWarning: The package pingouin is out of date. Your version is 0.5.0, the latest is 0.5.1.
Set the environment variable OUTDATED_IGNORE=1 to disable these warnings.
  return warn(


In [2]:
path = os.path.abspath(os.path.join(os.getcwd(), os.pardir))
subjects = [sub[:-3] for  sub in os.listdir(f"{path}/data/raw/")]

In [3]:
# Questionnaires
scales_df = pd.read_excel(f"{path}/data/scales.xlsx")
scales_df = scales_df.rename(columns={"Participant": "participant_id"})

# Summary

## Behaviors

In [4]:
hrd_df = pd.DataFrame({"participant_id": subjects})

hrd_df[["decision_rt_mean", "decision_rt_median", 
        "confidence_mean", "confidence_median", 
        "confidence_rt_mean", "confidence_rt_median",
        "psi_threshold", "psi_slope"]
      ] = "NaN"

In [5]:
for sub in hrd_df.participant_id.unique():

    # Load subject level behavior file
    try:
        behavior_df = pd.read_csv(f"{path}/data/raw/{sub}HRD/{sub}HRD_final.txt")
    except:
        print(f"Subject {sub}: file not found.")
        hrd_df = hrd_df[hrd_df.participant_id != sub]
        continue

    row_filter = (hrd_df.participant_id == sub) # Filter the group level df

    # Decision
    hrd_df.loc[row_filter, "decision_rt_mean"] = behavior_df.DecisionRT.dropna().mean()
    hrd_df.loc[row_filter, "decision_rt_median"] = behavior_df.DecisionRT.dropna().median()

    # Confidence rating
    hrd_df.loc[row_filter, "confidence_mean"] = behavior_df.Confidence.dropna().mean()
    hrd_df.loc[row_filter, "confidence_median"] = behavior_df.Confidence.dropna().median()
    hrd_df.loc[row_filter, "confidence_rt_mean"] = behavior_df.ConfidenceRT.dropna().mean()
    hrd_df.loc[row_filter, "confidence_rt_median"] = behavior_df.ConfidenceRT.dropna().median()

    # Psi estimates
    hrd_df.loc[row_filter, "psi_threshold"] = behavior_df.EstimatedThreshold.dropna().iloc[-1]
    hrd_df.loc[row_filter, "psi_slope"] = behavior_df.EstimatedSlope.dropna().iloc[-1]

Subject .ipynb_checkpoi: file not found.


In [6]:
hrd_df["participant_id"] = hrd_df["participant_id"].astype(int)
summary_df = pd.merge(scales_df, hrd_df, on=["participant_id"], how="right")
summary_df.to_csv(f"{path}/data/summary.csv", sep="\t", index=False)

## Metacognition

In [7]:
from multiprocessing import Process, Manager, cpu_count, Pool

num_processes = cpu_count()

print("Number of cpu available : ", num_processes)

Number of cpu available :  128


In [8]:
import arviz as az
import numpyro
import pandas as pd
from metadPy.utils import discreteRatings

numpyro.set_host_device_count(2)

def metad(input_data):

    ns = input_data[0]
    sub = input_data[1]

    # Load subject level behavior file
    try:
        behavior_df = pd.read_csv(f"{path}/data/raw/{sub}HRD/{sub}HRD_final.txt")

        # Drop trial with NaN in confidence rating
        behavior_df = behavior_df[~behavior_df.Confidence.isna()]

    except:
        print(f"Subject {sub}: data not found")

    # Ratings discretization
    try:
        new_ratings, _ = discreteRatings(
            behavior_df.Confidence.to_numpy(), verbose=False
        )
        behavior_df.loc[:, "discrete_confidence"] = new_ratings
    except ValueError:
        print(f"Subject {sub} - Cannot discretize ratings")

    # meta-d'
    try:
        # Code columns for Bayesian modelling
        behavior_df["Stimuli"] = behavior_df.copy()["Alpha"] > 0
        behavior_df["Responses"] = behavior_df["Decision"] == "More"
        behavior_df["Accuracy"] = (
            behavior_df["Stimuli"] & behavior_df["Responses"]
        ) | (~behavior_df["Stimuli"] & ~behavior_df["Responses"])

        # Fit meta-d' model
        results_df = behavior_df.hmetad(
            stimuli="Stimuli",
            accuracy="Accuracy",
            confidence="discrete_confidence",
            nRatings=4,
            padding=True,
            output="dataframe"
        )
        results_df["participant_id"] = sub
        ns.df = pd.concat([ns.df, results_df], ignore_index=True)
    except:
        print(f"Subject {sub} - Cannot perform meta-d modelling")

In [9]:
from itertools import repeat
import tqdm

num_partitions = hrd_df.participant_id.nunique()
num_processes = 20

mgr = Manager()
ns = mgr.Namespace()
ns.df = pd.DataFrame(columns={"participant_id", "d", "c", "meta_d", "m_ratio"})

pool = Pool(num_processes)

shared_arg = repeat(ns,num_partitions)

for _ in tqdm.tqdm(
    pool.map(
        metad, zip(shared_arg, subjects)
    ), total=num_partitions
):
    pass
pool.close()
pool.join()

/home/nicolas/.local/lib/python3.8/site-packages/metadPy/bayesian.py:356: RuntimeWarning: divide by zero encountered in double_scalars
  az.summary(traces, var_names="meta_d")["mean"]["meta_d"]


Subject 236 - Cannot discretize ratings
Subject 236 - Cannot perform meta-d modelling


  0%|          | 0/2000 [00:00<?, ?it/s]

  0%|          | 0/2000 [00:00<?, ?it/s]

/home/nicolas/.local/lib/python3.8/site-packages/metadPy/bayesian.py:356: RuntimeWarning: divide by zero encountered in double_scalars
  az.summary(traces, var_names="meta_d")["mean"]["meta_d"]
/home/nicolas/.local/lib/python3.8/site-packages/metadPy/bayesian.py:356: RuntimeWarning: divide by zero encountered in double_scalars
  az.summary(traces, var_names="meta_d")["mean"]["meta_d"]


...

  0%|          | 0/2000 [00:00<?, ?it/s]

  0%|          | 0/2000 [00:00<?, ?it/s]

  0%|          | 0/2000 [00:00<?, ?it/s]

  0%|          | 0/2000 [00:00<?, ?it/s]

UnboundLocalError: local variable 'behavior_df' referenced before assignment

In [10]:
# Retrieve metadpy results
metadpy_df = ns.df
metadpy_df = metadpy_df.replace('n/a', np.nan)
metadpy_df["participant_id"] = metadpy_df["participant_id"].astype(int)

In [11]:
summary_df = pd.merge(metadpy_df, summary_df, on=["participant_id"], how="right")

In [12]:
summary_df.to_csv(f"{path}/data/summary.csv", sep="\t", index=False)

# Heart rate variability

In [39]:
from systole.hrv import time_domain, frequency_domain, nonlinear_domain

In [217]:
summary_df = pd.read_csv(f"{path}/data/summary.csv", sep="\t")

In [218]:
summary_df.Enter_time = pd.to_datetime(summary_df.Date + " " +  summary_df.Enter_time)
summary_df.Exit_time = pd.to_datetime(summary_df.Date + " " +  summary_df.Exit_time)

In [219]:
summary_df["TaskDuration"] = summary_df.Exit_time - summary_df.Enter_time 
summary_df["TaskDuration"] = summary_df["TaskDuration"].dt.total_seconds()

In [279]:
hrv_df = pd.DataFrame([])
for sub in subjects:

    # Extract RR time serie
    try:
        rr = pd.read_csv(
            f"{path}/data/heartrate/" + [file for file in os.listdir(f"{path}/data/heartrate/") if file.startswith(f"P{sub} ")][0],
            skiprows=[2], header=3, sep=";"
        ).ArtifactCorrectedRRVector.to_numpy()

        # When the recording started
        start_recording = pd.to_datetime(
            pd.read_csv(
                f"{path}/data/heartrate/" + [file for file in os.listdir(f"{path}/data/heartrate/") if file.startswith(f"P{sub} ")][0],
                sep=";", nrows=1, 
            ).loc[0][0][12:],
            dayfirst=True
        )

    except IndexError:
        print(f"No data for {sub}")

    if np.any(rr > 6000):
        print(f"Artefacts in recordin for {sub}")
        continue

    start_delay = (summary_df[summary_df.participant_id == int(sub)].Enter_time - start_recording).dt.total_seconds().values[0]

    duration = summary_df[summary_df.participant_id == int(sub)].TaskDuration.values[0]

    # Remove pre-recording time
    rr = rr[(np.cumsum(rr) / 1000) > start_delay]

    # Remove post-recording time
    rr = rr[(np.cumsum(rr) / 1000) < start_delay]
    
    if len(rr) > 0:
        this_df = pd.concat([
            time_domain(rr=rr, input_type="rr_ms"),
            frequency_domain(rr=rr, input_type="rr_ms"),
            nonlinear_domain(rr=rr, input_type="rr_ms")
        ])

        this_df = pd.pivot_table(this_df, values='Values', columns='Metric')
        this_df["participant_id"] = sub

        hrv_df = pd.concat([hrv_df, this_df])
    
    else:
        print(f"No RR time series for {sub}")

Artefacts in recordin for 136
Artefacts in recordin for 30
No data for 31
No RR time series for 31
Artefacts in recordin for 4
Artefacts in recordin for 18
Artefacts in recordin for 184
No data for 42
No RR time series for 42
No data for 149
No RR time series for 149
No RR time series for 202
Artefacts in recordin for 217
Artefacts in recordin for 11
No data for 19
No RR time series for 19
Artefacts in recordin for 37
Artefacts in recordin for 156
Artefacts in recordin for 108
No data for .ipynb_checkpoi
Artefacts in recordin for .ipynb_checkpoi
Artefacts in recordin for 7


In [280]:
hrv_df["participant_id"] = hrv_df["participant_id"].astype(int)
summary_df = pd.merge(hrv_df, summary_df, on=["participant_id"], how="right")

In [282]:
summary_df.to_csv(f"{path}/data/summary.csv", sep="\t", index=False)